# 05 — SQL depth project (M6)

The analysis lives in numbered `sql/*.sql` files; this notebook only runs them and draws charts.
All queries read one frozen snapshot: `data/snapshots/jobs_2026-11-10.db` (12 scrape dates, 2026-06-30 → 2026-11-10).

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

# notebooks/ -> repo root on sys.path so `import jobskills` resolves.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from jobskills import config
from jobskills.sql_runner import connect, run

con = connect()

## Step 24 — Reconcile the snapshot against the source

Two independent read paths must agree: DuckDB reading the snapshot, and Python's `sqlite3` reading the live `jobs.db`.
The source is cut at `fetched_at < '2026-11-11'`, so this check stays valid after later weekly runs add rows.
A posting's `fetched_at` is its first fetch and never changes, so the rows before the cut are stable.

In [2]:
duck = run(con, "01_row_counts.sql")

# Same counts, written in SQLite, against the live source cut at the snapshot date.
SOURCE_COUNTS = """
WITH j AS (SELECT * FROM jobs WHERE fetched_at < '2026-11-11')
SELECT 'total' AS dimension, 'jobs' AS key, count(*) AS n FROM j
UNION ALL
SELECT 'total', 'skill_hits', count(*) FROM skill_hits WHERE posting_id IN (SELECT id FROM j)
UNION ALL
SELECT 'fetched_date', substr(fetched_at, 1, 10), count(*) FROM j GROUP BY 2
UNION ALL
SELECT 'source_region', coalesce(source_region, '(none)'), count(*) FROM j GROUP BY 2
"""
with sqlite3.connect(config.SQLITE_PATH) as src:
    source = pd.read_sql_query(SOURCE_COUNTS, src)

# Outer join so a key missing on either side shows up as NaN instead of vanishing.
recon = duck.merge(source, on=["dimension", "key"], how="outer", suffixes=("_duckdb", "_source"))
recon["match"] = recon["n_duckdb"] == recon["n_source"]
assert recon["match"].all(), recon[~recon["match"]]
recon

,dimension,key,n_duckdb,n_source,match
0,fetched_date,2026-06-30,1423,1423,True
1,fetched_date,2026-09-09,1276,1276,True
2,fetched_date,2026-09-14,321,321,True
3,fetched_date,2026-09-17,218,218,True
4,fetched_date,2026-09-22,346,346,True
5,fetched_date,2026-09-28,1,1,True
6,fetched_date,2026-10-05,765,765,True
7,fetched_date,2026-10-17,762,762,True
8,fetched_date,2026-10-19,146,146,True
9,fetched_date,2026-10-26,684,684,True


## Step 25 — Data-quality audit

Four checks, one `sql/02_audit_*.sql` file each. Each one reconciles back to the 7,182 snapshot rows.
- **Duplicates:** `dedup_key` (normalised company + title) collapses 2,609 surplus rows, mostly Adzuna re-posts.
- **Salary:** only 17% of rows have one, and almost all of them are Spanish.
- **Work mode:** `remote` is `unknown` for 90% of rows, because Adzuna descriptions are cut at about 500 characters.
- **Scrape calendar:** 9 weeks between 30 Jun and 9 Sep have no scrape, and the 28 Sep scrape found 1 posting.

In [ ]:
run(con, "02_audit_dedup.sql")

In [ ]:
run(con, "02_audit_salary.sql")

In [ ]:
run(con, "02_audit_remote.sql")

In [ ]:
run(con, "02_audit_scrape_calendar.sql")

## Step 26 — Swiss vs German vs Spanish skill demand

Share = distinct jobs mentioning the skill ÷ distinct jobs in that country (861 CH, 1,751 DE, 1,974 ES).
Compare the **ranks**, not the shares: Adzuna cuts descriptions at 500 chars, and the share of jobs with any skill hit differs by country (CH 45%, DE 34%, ES 48%).
`r` is excluded because its matcher mostly hits "R&D". Below the top ~12 skills, counts are small (under 15 jobs per country).

In [ ]:
demand = run(con, "03_country_skill_demand.sql")
demand.head(15)